In [1]:
# ── Setup ────────────────────────────────────────────────────────────────
!pip install -q datasets transformers openpyxl

import re
import json
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

pd.set_option("display.max_colwidth", 80)


In [2]:
# ── 1b. Sindhi Wikipedia — formal / encyclopedic ───────────────────────────
wiki = load_dataset("wikimedia/wikipedia", "20231101.sd", split="train")

# Sample ~2000 articles, then split into sentences for a fair per-sentence
# comparison against the other registers (crude splitter — refine later
# with a proper Sindhi/Arabic-script sentence segmenter if needed).
SAMPLE_N = 2000
wiki_sample = wiki.shuffle(seed=42).select(range(min(SAMPLE_N, len(wiki))))

def split_sentences(text):
    # Sindhi uses Arabic-script punctuation; split on ۔ . ! ? and newlines
    parts = re.split(r"[۔\.!؟\n]+", text)
    return [p.strip() for p in parts if len(p.strip()) > 3]

wiki_sentences = []
for row in wiki_sample:
    wiki_sentences.extend(split_sentences(row["text"]))

print(f"Sindhi Wikipedia: {len(wiki_sample)} articles -> {len(wiki_sentences)} sentences")
print(wiki_sentences[0] if wiki_sentences else "(none)")


README.md: 0.00B [00:00, ?B/s]

20231101.sd/train-00000-of-00001.parquet:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16928 [00:00<?, ? examples/s]

Sindhi Wikipedia: 2000 articles -> 33512 sentences
ابن سعد- جو پورو نالو ابو عبدالله محمد -بن- سعد -بن- منيع البصري هو ۽ 168هه/ 784ع ۾ بصري ۾ ڄائو


# Custom Sindhi tokenizer — training only

Standalone notebook: builds the training corpus and trains the custom
SentencePiece Unigram (+ byte_fallback) tokenizer at 4 vocab sizes, without
downloading XLM-RoBERTa/mT5/Qwen/BLOOM or running the full 11-tokenizer
comparison — that lives in `sindhi-tokenizer-benchmark-stage3.ipynb`.

**Design:**
- Unigram + `byte_fallback=True`, `character_coverage=1.0`
- ~85% Sindhi / ~15% Urdu+Persian+Arabic by character count
- Vocab sizes: 32K / 64K / 100K / 150K
- Training corpus kept disjoint from the eval registers used in the benchmark
  notebook (same seed=42 Wikipedia shuffle, training starts after the first
  2000 articles reserved for eval)

**After this notebook finishes:** download the `spm_train/` folder (or the
individual `.model` files) from Kaggle's output panel, then either (a) upload
them into the benchmark notebook's environment and run cells 10i onward
there to get the full comparison table, or (b) run the lightweight sanity
check at the end of this notebook for a quick look without the full
benchmark.

In [3]:
# ── 10a. Setup for tokenizer training ──────────────────────────────────────
!pip install -q sentencepiece

import os
import sentencepiece as spm
from transformers import LlamaTokenizer

os.makedirs("spm_train", exist_ok=True)

In [4]:
# ── 10b. Wikipedia TRAINING slice — disjoint from the eval sample (cell 1b) ──
# Cell 1b used wiki.shuffle(seed=42).select(range(SAMPLE_N)) for evaluation.
# Re-using the same shuffle+seed and taking the NEXT slice guarantees zero
# overlap with the eval sentences — training on the eval set would invalidate
# every fertility/single-token comparison later in this notebook.
TRAIN_SAMPLE_N = 6000
wiki_shuffled = wiki.shuffle(seed=42)
wiki_train_end = min(SAMPLE_N + TRAIN_SAMPLE_N, len(wiki_shuffled))
wiki_train_sample = wiki_shuffled.select(range(SAMPLE_N, wiki_train_end))

wiki_train_sentences = []
for row in wiki_train_sample:
    wiki_train_sentences.extend(split_sentences(row["text"]))

print(f"Wikipedia TRAIN slice: {len(wiki_train_sample)} articles -> {len(wiki_train_sentences)} sentences")
print(f"(disjoint from the {SAMPLE_N}-article eval sample used in section 1 — zero overlap by construction)")

Wikipedia TRAIN slice: 6000 articles -> 104775 sentences
(disjoint from the 2000-article eval sample used in section 1 — zero overlap by construction)


In [5]:
# ── 10c. Informal/social training text — NOT the Sentiment eval dataset ────
import pandas as pd
from urllib.parse import quote

SINDHI_NLP_BASE = "https://raw.githubusercontent.com/Sindhi-NLP/sindhi-NLP-dataset/main"
SINDHI_NLP_FILES = {
    "twitter_charooxyz": (f"{SINDHI_NLP_BASE}/Twitter-Data/charooxyz-page-tweets.csv", "tweet"),
    "twitter_amarfayazburiro": (f"{SINDHI_NLP_BASE}/Twitter-Data/amarfayazburiro-page-tweets.csv", "Tweet"),
    "awamiawaz_articles": (f"{SINDHI_NLP_BASE}/{quote('awamiawaz articles classification dataset/Articles Dataset.csv')}", "article"),
    "daily_kawish_articles": (f"{SINDHI_NLP_BASE}/{quote('daily kawish articles/daily kawish article dataset.csv')}", "article"),
}

informal_train_sentences = []
for name, (url, col) in SINDHI_NLP_FILES.items():
    try:
        df = pd.read_csv(url)
        texts = df[col].dropna().astype(str).tolist()
        for t in texts:
            informal_train_sentences.extend(split_sentences(t))
        print(f"loaded {name:24s} {len(texts):>5} rows -> running total {len(informal_train_sentences)} sentences")
    except Exception as e:
        print(f"FAILED to load {name}: {e}")

print(f"\nTotal informal/news training sentences: {len(informal_train_sentences)}")

loaded twitter_charooxyz         2046 rows -> running total 2153 sentences
loaded twitter_amarfayazburiro   1791 rows -> running total 5675 sentences
loaded awamiawaz_articles        3363 rows -> running total 38351 sentences
loaded daily_kawish_articles     6156 rows -> running total 363164 sentences

Total informal/news training sentences: 363164


In [6]:
# ── 10d. Contemporary/mixed Sindhi text — modern-vocabulary coverage ───────
# No dedicated "Sindhi tech vocabulary" dataset exists, so this is the
# closest available proxy: a recent (~June 2026) general Sindhi corpus that
# postdates every benchmarked tokenizer's training cutoff. Column names are
# not documented publicly, so detect them the same defensive way cell 1c
# already does for the Sentiment dataset — don't assume a schema.
contemporary_train_sentences = []
try:
    contemporary_ds = load_dataset("shaikhsalman/sindhi-llm-corpus", split="train")
    candidate_cols = [c for c in contemporary_ds.column_names if c.lower() in ("text", "content", "sentence", "article")]
    text_col = candidate_cols[0] if candidate_cols else contemporary_ds.column_names[0]
    print(f"shaikhsalman/sindhi-llm-corpus columns: {contemporary_ds.column_names} (using '{text_col}')")

    # Cap it — this corpus is large (291K rows); a sample is enough to add
    # modern-vocabulary signal without dominating the training mix.
    CONTEMPORARY_SAMPLE_N = 20000
    contemporary_sample = contemporary_ds.shuffle(seed=7).select(range(min(CONTEMPORARY_SAMPLE_N, len(contemporary_ds))))
    for row in contemporary_sample:
        contemporary_train_sentences.extend(split_sentences(str(row[text_col])))
    print(f"Contemporary corpus: {len(contemporary_sample)} rows -> {len(contemporary_train_sentences)} sentences")
except Exception as e:
    print(f"Could not load shaikhsalman/sindhi-llm-corpus ({e}) — continuing without it; "
          f"the tech-loanword target in the results table below may not be reachable without a substitute source.")

README.md:   0%|          | 0.00/418 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/221M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/222M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/3.06M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/291397 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1465 [00:00<?, ? examples/s]

shaikhsalman/sindhi-llm-corpus columns: ['text', 'source'] (using 'text')
Contemporary corpus: 20000 rows -> 630180 sentences


In [7]:
# ── 10e. Urdu / Persian / Arabic — the ~15% cross-lingual slice ────────────
# Reusing the same wikimedia/wikipedia loader already proven to work for
# Sindhi in cell 1b, just with different language configs.
CROSS_LINGUAL_CONFIGS = {"urdu": "20231101.ur", "persian": "20231101.fa", "arabic": "20231101.ar"}
CROSS_LINGUAL_SAMPLE_N = 1500  # per language — kept small; this is a minority-weighted slice

cross_lingual_sentences = []
for lang, config in CROSS_LINGUAL_CONFIGS.items():
    try:
        lang_ds = load_dataset("wikimedia/wikipedia", config, split="train")
        lang_sample = lang_ds.shuffle(seed=42).select(range(min(CROSS_LINGUAL_SAMPLE_N, len(lang_ds))))
        lang_sentences = []
        for row in lang_sample:
            lang_sentences.extend(split_sentences(row["text"]))
        cross_lingual_sentences.extend(lang_sentences)
        print(f"{lang:8s} ({config}): {len(lang_sample)} articles -> {len(lang_sentences)} sentences")
    except Exception as e:
        print(f"FAILED to load {lang} ({config}): {e}")

print(f"\nTotal cross-lingual (Urdu+Persian+Arabic) sentences: {len(cross_lingual_sentences)}")

20231101.ur/train-00000-of-00001.parquet:   0%|          | 0.00/168M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/200154 [00:00<?, ? examples/s]

urdu     (20231101.ur): 1500 articles -> 29203 sentences


20231101.fa/train-00000-of-00004.parquet:   0%|          | 0.00/298M [00:00<?, ?B/s]

20231101.fa/train-00001-of-00004.parquet:   0%|          | 0.00/157M [00:00<?, ?B/s]

...

20231101.ar/train-00004-of-00007.parquet:   0%|          | 0.00/65.1M [00:00<?, ?B/s]

20231101.ar/train-00005-of-00007.parquet:   0%|          | 0.00/156M [00:00<?, ?B/s]

20231101.ar/train-00006-of-00007.parquet:   0%|          | 0.00/246M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1219201 [00:00<?, ? examples/s]

arabic   (20231101.ar): 1500 articles -> 32619 sentences

Total cross-lingual (Urdu+Persian+Arabic) sentences: 89449


In [8]:
# ── 10f. Clitic list — kept for training-time atomicity, NOT for preprocessing ──
# Empirically tested (see diagnostic run): جو/جي/تي/۾/کي/سان are already
# orthographically space-separated in this corpus. A fused-word check found
# apparent "fusion" for تي/سان/جي was entirely coincidental letter-overlap
# with unrelated words (صدارتي, انسان, پنهنجي, etc.) or different grammatical
# morphemes (verb-subjunctive -جي) — not the postposition. ۾'s 0.2% rate
# (a distinctive character with no plausible coincidental overlap) confirms
# this as the true baseline.
#
# Conclusion: no text preprocessing needed or applied. Atomicity for these
# six tokens is guaranteed a different way — via user_defined_symbols at
# training time (cell 10h) — which needs no matching step at inference time,
# unlike a preprocessing function would have.
CLITICS = ["جو", "جي", "تي", "۾", "کي", "سان"]

In [9]:
# ── 10g. Assemble the weighted training corpus (85/15, by character count) ─
# Sindhi-side sources: disjoint Wikipedia slice + informal/news text +
# contemporary corpus. Cross-lingual side: Urdu/Persian/Arabic.
# No clitic pre-marking applied — confirmed unnecessary (see cell 10f).
# Ratio is validated by character count, not sentence count, since sentence
# length varies a lot across these sources (tweets vs. news articles).
TARGET_SINDHI_RATIO = 0.85

sindhi_train_sentences = (
    wiki_train_sentences
    + informal_train_sentences
    + contemporary_train_sentences
)
sindhi_chars = sum(len(s) for s in sindhi_train_sentences)
cross_lingual_chars = sum(len(s) for s in cross_lingual_sentences)

print(f"Sindhi-side:        {len(sindhi_train_sentences)} sentences, {sindhi_chars} chars")
print(f"Cross-lingual-side: {len(cross_lingual_sentences)} sentences, {cross_lingual_chars} chars")
print(f"Current Sindhi share: {sindhi_chars / (sindhi_chars + cross_lingual_chars):.1%}  (target: {TARGET_SINDHI_RATIO:.0%})")

# Trim or oversample the cross-lingual side to hit the target ratio exactly.
target_cross_lingual_chars = sindhi_chars * (1 - TARGET_SINDHI_RATIO) / TARGET_SINDHI_RATIO
if cross_lingual_chars > target_cross_lingual_chars:
    kept, running = [], 0
    for s in cross_lingual_sentences:
        if running >= target_cross_lingual_chars:
            break
        kept.append(s); running += len(s)
    cross_lingual_final = kept
else:
    # oversample (repeat) to reach the target share
    reps = max(1, round(target_cross_lingual_chars / max(cross_lingual_chars, 1)))
    cross_lingual_final = cross_lingual_sentences * reps

final_chars_sindhi = sindhi_chars
final_chars_cross = sum(len(s) for s in cross_lingual_final)
print(f"\nFinal mix -> Sindhi: {final_chars_sindhi / (final_chars_sindhi + final_chars_cross):.1%}  "
      f"Cross-lingual: {final_chars_cross / (final_chars_sindhi + final_chars_cross):.1%}")

TRAINING_CORPUS_PATH = "spm_train/corpus.txt"
with open(TRAINING_CORPUS_PATH, "w", encoding="utf-8") as f:
    for s in sindhi_train_sentences + cross_lingual_final:
        s = s.strip()
        if s:
            f.write(s + "\n")

print(f"\nWrote training corpus: {TRAINING_CORPUS_PATH} "
      f"({len(sindhi_train_sentences) + len(cross_lingual_final)} lines)")

Sindhi-side:        1098119 sentences, 102682270 chars
Cross-lingual-side: 89449 sentences, 4933682 chars
Current Sindhi share: 95.4%  (target: 85%)

Final mix -> Sindhi: 83.9%  Cross-lingual: 16.1%

Wrote training corpus: spm_train/corpus.txt (1455915 lines)


NameError: name 'args' is not defined

In [19]:
%%writefile train_one_spm.py
import argparse
import sentencepiece as spm

parser = argparse.ArgumentParser()
parser.add_argument("--input", required=True)
parser.add_argument("--prefix", required=True)
parser.add_argument("--vocab_size", type=int, required=True)
parser.add_argument("--user_defined_symbols", required=True)
args = parser.parse_args()

spm.SentencePieceTrainer.train(
    input=args.input,
    model_prefix=args.prefix,
    vocab_size=args.vocab_size,
    model_type="unigram",
    byte_fallback=True,
    character_coverage=1.0,
    user_defined_symbols=args.user_defined_symbols.split(",") if args.user_defined_symbols else [],
    unk_id=0, bos_id=1, eos_id=2, pad_id=-1,
    unk_piece="<unk>", bos_piece="<s>", eos_piece="</s>",
)
print("TRAINING_COMPLETE")

Overwriting train_one_spm.py


In [20]:
# ── 10h. Train SentencePiece Unigram + byte_fallback across vocab sizes ────
# FORCED_WHOLE_TOKENS empirically confirmed unnecessary and harmful: plain
# training already produces single-token '▁جي', '▁تي', '▁ڪمپيوٽر', etc., and
# forcing them (either bare or space-prefixed) measurably hurt the training
# objective (obj≈9.6 unforced vs obj≈14.9 forced, same vocab size, same
# corpus). CLITICS itself stays defined for documentation purposes only.
import subprocess
import time
import os

FORCED_WHOLE_TOKENS = []
VOCAB_SIZES = [32000, 64000, 100000, 150000]
CUSTOM_MODEL_PREFIX = "spm_train/sindhi_custom"

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

overall_start = time.time()
for vs in VOCAB_SIZES:
    prefix = f"{CUSTOM_MODEL_PREFIX}_{vs}"
    vs_start = time.time()
    print(f"\n{'='*70}\nTraining vocab_size={vs} ... (started {time.strftime('%H:%M:%S')})\n{'='*70}")

    cmd = [
        "python3", "train_one_spm.py",
        f"--input={TRAINING_CORPUS_PATH}",
        f"--prefix={prefix}",
        f"--vocab_size={vs}",
        f"--user_defined_symbols={','.join(FORCED_WHOLE_TOKENS) if FORCED_WHOLE_TOKENS else ''}",
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()

    if proc.returncode != 0:
        raise RuntimeError(f"Training failed for vocab_size={vs} (exit code {proc.returncode})")

    vocab_path = f"{prefix}.vocab"
    with open(vocab_path, encoding="utf-8") as f:
        n_lines = sum(1 for _ in f)
    if n_lines < vs * 0.9:
        raise RuntimeError(f"vocab_size={vs}: .vocab file only has {n_lines} lines — training likely failed")

    vs_elapsed = time.time() - vs_start
    print(f"  -> {prefix}.model / {prefix}.vocab  ({n_lines} pieces, done in {vs_elapsed/60:.1f} min)")

overall_elapsed = time.time() - overall_start
print(f"\nAll {len(VOCAB_SIZES)} vocab sizes trained in {overall_elapsed/60:.1f} min total.")


Training vocab_size=32000 ... (started 19:25:56)
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: spm_train/corpus.txt
  input_format: 
  model_prefix: spm_train/sindhi_custom_32000
  model_type: UNIGRAM
  vocab_size: 32000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  p

In [21]:
# 10h-init — reset the dicts first, don't let old entries linger
tokenizers = {}
tokenizer_vocab_sizes = {}

In [22]:
# ── 10i. Load trained models — direct SentencePieceProcessor, not LlamaTokenizer ──
# LlamaTokenizer's custom protobuf-patching logic (added for an unrelated
# Llama-specific bug) appears to mishandle byte_fallback + custom
# user_defined_symbols, silently loading a near-empty 3-token model despite
# the underlying .model/.vocab files being confirmed correct on disk.
# SentencePieceProcessor loads the raw file with no reinterpretation.
import sentencepiece as spm

class SPMTokenizerAdapter:
    def __init__(self, model_path):
        self.sp = spm.SentencePieceProcessor(model_file=model_path)
        self.unk_token = "<unk>"

    @property
    def vocab_size(self):
        return self.sp.get_piece_size()

    def encode(self, text, add_special_tokens=False):
        return self.sp.encode(text, out_type=int)

    def convert_ids_to_tokens(self, ids):
        return [self.sp.id_to_piece(i) for i in ids]

    def convert_tokens_to_string(self, tokens):
        return self.sp.decode(tokens)

tokenizers = {}
tokenizer_vocab_sizes = {}
custom_tokenizer_names = []
for vs in VOCAB_SIZES:
    name = f"CustomSindhi_{vs // 1000}k"
    model_path = f"{CUSTOM_MODEL_PREFIX}_{vs}.model"
    tok = SPMTokenizerAdapter(model_path)
    tokenizers[name] = tok
    tokenizer_vocab_sizes[name] = tok.vocab_size
    custom_tokenizer_names.append(name)
    print(f"loaded {name:18s} <- {model_path}  (vocab_size={tok.vocab_size})")

loaded CustomSindhi_32k   <- spm_train/sindhi_custom_32000.model  (vocab_size=32000)
loaded CustomSindhi_64k   <- spm_train/sindhi_custom_64000.model  (vocab_size=64000)
loaded CustomSindhi_100k  <- spm_train/sindhi_custom_100000.model  (vocab_size=100000)
loaded CustomSindhi_150k  <- spm_train/sindhi_custom_150000.model  (vocab_size=150000)


In [23]:
# ── Quick sanity check (no full benchmark needed) ──────────────────────────
# A handful of hand-picked words covering the cases this design specifically
# targets: an aspirated digraph, a tech loanword, and a clitic.
SANITY_WORDS = ["ٻارن", "ڪمپيوٽر", "پاڻي", "جي", "ننڍڙو"]

for vs in VOCAB_SIZES:
    name = f"CustomSindhi_{vs // 1000}k"
    tok = tokenizers[name]
    print(f"--- {name} (vocab_size={tok.vocab_size}) ---")
    for w in SANITY_WORDS:
        ids = tok.encode(w, add_special_tokens=False)
        pieces = tok.convert_ids_to_tokens(ids)
        print(f"  {w:10s} -> {pieces}  ({len(pieces)} token{'s' if len(pieces) != 1 else ''})")
    print()

print("Full comparison against the other 10 tokenizers happens in "
      "sindhi-tokenizer-benchmark-stage3.ipynb, cells 10j-10k.")

--- CustomSindhi_32k (vocab_size=32000) ---
  ٻارن       -> ['▁ٻارن']  (1 token)
  ڪمپيوٽر    -> ['▁ڪمپيوٽر']  (1 token)
  پاڻي       -> ['▁پاڻي']  (1 token)
  جي         -> ['▁جي']  (1 token)
  ننڍڙو      -> ['▁ننڍڙو']  (1 token)

--- CustomSindhi_64k (vocab_size=64000) ---
  ٻارن       -> ['▁ٻارن']  (1 token)
  ڪمپيوٽر    -> ['▁ڪمپيوٽر']  (1 token)
  پاڻي       -> ['▁پاڻي']  (1 token)
  جي         -> ['▁جي']  (1 token)
  ننڍڙو      -> ['▁ننڍڙو']  (1 token)

--- CustomSindhi_100k (vocab_size=100000) ---
  ٻارن       -> ['▁ٻارن']  (1 token)
  ڪمپيوٽر    -> ['▁ڪمپيوٽر']  (1 token)
  پاڻي       -> ['▁پاڻي']  (1 token)
  جي         -> ['▁جي']  (1 token)
  ننڍڙو      -> ['▁ننڍڙو']  (1 token)

--- CustomSindhi_150k (vocab_size=150000) ---
  ٻارن       -> ['▁ٻارن']  (1 token)
  ڪمپيوٽر    -> ['▁ڪمپيوٽر']  (1 token)
  پاڻي       -> ['▁پاڻي']  (1 token)
  جي         -> ['▁جي']  (1 token)
  ننڍڙو      -> ['▁ننڍڙو']  (1 token)

Full comparison against the other 10 tokenizers happens in sindhi-to

In [ ]:
# ── Quick check: do these words reach single-token status naturally? ──────
import sentencepiece as spm

for label, symbols in [("no_forcing", []), ("space_prefixed", CLITICS + ["ڪمپيوٽر", "انٽرنيٽ"] + ["▁" + w for w in CLITICS + ["ڪمپيوٽر", "انٽرنيٽ"]])]:
    prefix = f"spm_train/test_{label}_32000"
    spm.SentencePieceTrainer.train(
        input=TRAINING_CORPUS_PATH, model_prefix=prefix, vocab_size=32000,
        model_type="unigram", byte_fallback=True, character_coverage=1.0,
        user_defined_symbols=symbols,
        unk_id=0, bos_id=1, eos_id=2, pad_id=-1,
        unk_piece="<unk>", bos_piece="<s>", eos_piece="</s>",
    )
    sp = spm.SentencePieceProcessor(model_file=f"{prefix}.model")
    print(f"--- {label} ---")
    for w in ["ڪمپيوٽر", "جي", "تي"]:
        pieces = [sp.id_to_piece(i) for i in sp.encode(w, out_type=int)]
        print(f"  {w:10s} -> {pieces}")